# Example 9.6.1 / 1: Solve One Instance per File

Companion notebook for **9.6.1 Repeated Ingestion and Data Reshaping** in the AMPL lab Documentation, section "Solve One Instance per File". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve One Instance per File

A symbolic parameter can hold the file name, and a parenthesized expression in the declaration
uses it, so one declaration reads whichever file the parameter names. Here `week1.csv` has all
three products and `week2.csv` only two:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param infile symbolic;            # the file to read next
set PROD;
param rate {PROD} > 0;
param profit {PROD};
param market {PROD} >= 0;
param avail >= 0 default 40;

var Make {p in PROD} >= 0, <= market[p];
maximize TotalProfit: sum {p in PROD} profit[p] * Make[p];
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

table Week IN "amplcsv" (infile):
    PROD <- [product], rate, profit, market;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
with open("week1.csv", "w") as f:
    f.write("product,rate,profit,market\n"
            "bands,200,25,6000\ncoils,140,30,4000\nplate,160,29,3500\n")
with open("week2.csv", "w") as f:
    f.write("product,rate,profit,market\n"
            "bands,200,25,3000\nplate,160,29,5000\n")

In [5]:
# run.py
ampl.eval("load amplcsv.dll;")
for name in ["week1.csv", "week2.csv"]:
    ampl.eval("reset data;")
    ampl.param["infile"] = name
    ampl.read_table("Week")
    ampl.solve(solver="highs", verbose=False)
    if ampl.solve_result != "solved":
        raise RuntimeError(f"{name}: {ampl.solve_result}")
    print(name, ampl.set["PROD"].to_list(), ampl.obj["TotalProfit"].value())

week1.csv ['bands', 'coils', 'plate'] 196400.0
week2.csv ['bands', 'plate'] 191000.0


### Pure AMPL: model.run

`reset data;` (sent with `ampl.eval`, as amplpy has no method for it) clears every set and
parameter before the next file, so nothing from `week1.csv` survives into the second instance.
A new `AMPL()` object per file gives the same isolation at the cost of reading the model again;
Resetting Models and Data (AMPL lab documentation, section 3.7.1) compares the reset commands.

Pure AMPL, with the same `model.mod` and files:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
with open("week1.csv", "w") as f:
    f.write("product,rate,profit,market\n"
            "bands,200,25,6000\ncoils,140,30,4000\nplate,160,29,3500\n")
with open("week2.csv", "w") as f:
    f.write("product,rate,profit,market\n"
            "bands,200,25,3000\nplate,160,29,5000\n")

In [8]:
%%ampl_eval
# model.run
load amplcsv.dll;
option solver highs;
option solver_msg 0;
for {name in {"week1.csv", "week2.csv"}} {
    reset data;
    let infile := name;
    read table Week;
    solve;
    printf "%s %d products, profit %g\n", infile, card(PROD), TotalProfit;
}

week1.csv 3 products, profit 196400
week2.csv 2 products, profit 191000
